# DATATHON 2026 — THE GRIDBREAKER
**Hosted by:** VinTelligence - VinUniversity Data Science & AI Club

---

## THÔNG TIN NHÓM
- **Tên đội thi**: Last Dance
- **Mã đội thi**: kVmzJpHWUFT6mv82wG4U
- **Thành viên**: 

| STT | Họ và tên        |  Vai trò   | Email |
| --- | ---------------- | ---------- | ----- |
| 1   | Bùi Lê Khôi      | Đội trưởng |       |
| 2   | Nguyễn Hà Anh    | Thành viên |       |
| 3   | Bùi Công Mậu     | Thành viên |       |
| 4   | Thái Hữu Thọ     | Thành viên |       |

---

## GIỚI THIỆU FILE NOTEBOOK

- **Mục đích**: Báo cáo bài làm cho **Phần 1 — Câu hỏi Trắc nghiệm**

- **Nội dung trong file**:
    - Bài giải tự luận chi tiết cho các câu hỏi trong phần 1.
    - Code solution (nếu có) để giải các câu hỏi này.

- **Dữ liệu sử dụng**:
    - Dữ liệu được cung cấp bởi ban tổ chức, gồm các file csv được chia theo lớp.
    - Bài làm sử dụng trực tiếp dữ liệu thô, không tiền xử lý.
    
---

Bài làm được trình bày từ sau dòng này.

---
---

### Import các libs và dependencies

In [2]:
import re
import os
import math

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl as pxl
import seaborn as sns
import statsmodels.api as sm
import datetime as dt

from datetime import timedelta
from scipy import stats
from math import sqrt

from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, pairwise_distances, silhouette_score, precision_score, recall_score, f1_score
from sklearn.metrics import RocCurveDisplay, roc_auc_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

import warnings
warnings.filterwarnings("ignore")

# Ensure plots appear in the notebook
%matplotlib inline

# Print last updated timestamp
import time
print(f"Cập nhật lần cuối vào thời điểm {time.asctime()}")

Cập nhật lần cuối vào thời điểm Wed Apr 22 08:12:12 2026


---

### Câu 1
> ***Q1**. Trong số các khách hàng có nhiều hơn một đơn hàng, **trung vị số ngày giữa hai lần mua liên tiếp** (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ ```orders.csv```)*
>
> A) 30 ngày
>
> B) 90 ngày
>
> C) 180 ngày
>
> D) 365 ngày

**Phân tích**:

Để tính toán, ta cần chọn ra các khách hàng xuất hiện nhiều hơn 1 lần. Mỗi khách hàng, ta sort các đơn theo thứ tự thời gian, sau đó lấy các giá trị chênh lệch số ngày giữa mỗi 2 đơn liên tiếp. Cuối cùng, sort tất cả các giá trị thu đc theo thứ tự không giảm để tính trung vị.

In [3]:
# Đọc data orders.csv
df_orders = pd.read_csv('../data/raw/orders.csv')
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   order_id        646945 non-null  int64 
 1   order_date      646945 non-null  object
 2   customer_id     646945 non-null  int64 
 3   zip             646945 non-null  int64 
 4   order_status    646945 non-null  object
 5   payment_method  646945 non-null  object
 6   device_type     646945 non-null  object
 7   order_source    646945 non-null  object
dtypes: int64(3), object(5)
memory usage: 39.5+ MB


In [4]:
# Chuyển đổi order_date: object -> datetime
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])

# Sort dữ liệu theo customer_id và order_date để tính đúng theo trình tự thời gian
df_orders = df_orders.sort_values(by=['customer_id', 'order_date'])

# Nhóm theo từng khách hàng, sau đó tính khoảng cách giữa các lần mua (Inter-order gap)
df_orders['inter_order_gap'] = df_orders.groupby('customer_id')['order_date'].diff()

# Convert kết quả từ timedelta sang int (số ngày)
df_orders['inter_order_gap_days'] = df_orders['inter_order_gap'].dt.days

# Lọc NaN để loại các khách 1 đơn, sau đó lấy cột số ngày giữa các lần mua liên tiếp rồi tính median
all_gaps = df_orders['inter_order_gap_days'].dropna()
median_gap = all_gaps.median()

print(f"Số gap thu được: {len(all_gaps)}")
print(f"Trung vị số ngày giữa hai lần mua liên tiếp: {median_gap} ngày")

Số gap thu được: 556699
Trung vị số ngày giữa hai lần mua liên tiếp: 144.0 ngày


Dựa theo kết quả thu được, đáp án xấp xỉ gần nhất là **C) 180 ngày**.

---

### Câu 2
> ***Q2**. Phân khúc sản phẩm (```segment```) nào trong ```products.csv``` có **tỷ suất lợi nhuận gộp trung bình** cao nhất, với công thức ```(price − cogs)/price```?*
>
> A) Premium
>
> B) Performance
>
> C) Activewear
>
> D) Standard

**Phân tích**:

Ta có công thức ```GrossMargin = (price − cogs)/price```, với ràng buộc ```cogs < price```.

Để giải quyết bài toán, ta tính tỷ suất lợi nhuận gộp cho từng sản phẩm, sau đó gom nhóm theo từng segment để lấy trung bình cộng. Sắp xếp lại kết quả, ta có bảng margin theo segment, và từ đó biết được segment nào có tỷ suất lợi nhuận gộp cao nhất.

In [5]:
# Đọc data products.csv
df_products = pd.read_csv('../data/raw/products.csv')
df_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    2412 non-null   int64  
 1   product_name  2412 non-null   object 
 2   category      2412 non-null   object 
 3   segment       2412 non-null   object 
 4   size          2412 non-null   object 
 5   color         2412 non-null   object 
 6   price         2412 non-null   float64
 7   cogs          2412 non-null   float64
dtypes: float64(2), int64(1), object(5)
memory usage: 150.9+ KB


In [25]:
# Tính tỷ suất lợi nhuận gộp (gross margin) cho từng sản phẩm theo công thức đã cho
df_products['gross_margin'] = (df_products['price'] - df_products['cogs']) / df_products['price']

# Sau đó gom theo segment và tính trung bình
segment_margin_avg = df_products.groupby('segment')['gross_margin'].mean()

# Sắp xếp kết quả từ cao xuống thấp
best_segment = segment_margin_avg.sort_values(ascending=False)

print("Bảng tỉ suất lợi nhuận gộp trung bình theo segment:")
print(best_segment)
print(f"Segment có tỷ suất lợi nhuận gộp trung bình cao nhất: {best_segment.index[0]},")
print(f"giá trị là {best_segment.iloc[0]:.2%}")

Bảng tỉ suất lợi nhuận gộp trung bình theo segment:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64
Segment có tỷ suất lợi nhuận gộp trung bình cao nhất: Standard,
giá trị là 31.34%


Dựa theo kết quả thu được, đáp án cần chọn là **D) Standard**.

---

### Câu 3
> ***Q3**. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục **Streetwear** (**join** ```returns``` với ```products``` theo ```product_id```), **lý do trả hàng** nào xuất hiện nhiều nhất?*
>
> A) ```defective```
>
> B) ```wrong_size```
>
> C) ```changed_mind```
>
> D) ```not_as_described```

**Phân tích**:

Ta cần join 2 bảng ```returns``` và ```products``` theo ```product_id```. Khi đó, ta lọc lấy các dòng có category là ```Streetwear``` và đếm số lần xuất hiện của từng lý do trả hàng. Sắp xếp lại số lần xuất hiện, ta sẽ tìm được lý do trả hàng phổ biến nhất.

In [7]:
# Đọc data returns.csv
df_returns = pd.read_csv('../data/raw/returns.csv')

# Merge với products theo product_id
df_returns_products = pd.merge(df_returns, df_products, on='product_id', how='inner')

df_returns_products.info()

# In ra các giá trị unique của cột category và return_reason để kiểm tra
print("\nCác giá trị unique trong cột 'category':")
print(df_returns_products['category'].unique())
print("\nCác giá trị unique trong cột 'return_reason':")
print(df_returns_products['return_reason'].unique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   return_id        39939 non-null  object 
 1   order_id         39939 non-null  int64  
 2   product_id       39939 non-null  int64  
 3   return_date      39939 non-null  object 
 4   return_reason    39939 non-null  object 
 5   return_quantity  39939 non-null  int64  
 6   refund_amount    39939 non-null  float64
 7   product_name     39939 non-null  object 
 8   category         39939 non-null  object 
 9   segment          39939 non-null  object 
 10  size             39939 non-null  object 
 11  color            39939 non-null  object 
 12  price            39939 non-null  float64
 13  cogs             39939 non-null  float64
 14  gross_margin     39939 non-null  float64
dtypes: float64(4), int64(3), object(8)
memory usage: 4.6+ MB

Các giá trị unique trong cột 'category':
['Str

In [27]:
# Lọc các bản ghi thuộc danh mục 'Streetwear'
streetwear_returns = df_returns_products[df_returns_products['category'] == 'Streetwear']

# Đếm số lần xuất hiện của từng lý do trả hàng
reason_counts = streetwear_returns['return_reason'].value_counts()

print("Thống kê các lý do trả hàng trong danh mục Streetwear:")
print(reason_counts)
print(f"Lý do trả hàng phổ biến nhất trong Streetwear: {reason_counts.idxmax()},")
print(f"với {reason_counts.max()} lần trả hàng")

Thống kê các lý do trả hàng trong danh mục Streetwear:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64
Lý do trả hàng phổ biến nhất trong Streetwear: wrong_size,
với 7626 lần trả hàng


Dựa theo kết quả thu được, đáp án cần chọn là **B) ```wrong_size```**.

---

### Câu 4
> ***Q4**. Trong ```web_traffic.csv```, nguồn truy cập (```traffic_source```) nào có **tỷ lệ thoát trung bình** (```bounce_rate```) **thấp nhất** trên tất cả các ngày xuất hiện nguồn đó trong cột ```traffic_source```?*
>
> A) ```organic_search```
>
> B) ```paid_search```
>
> C) ```email_campaign```
> 
> D) ```social_media```

**Phân tích**:

Với câu này thì chỉ cần làm y như đề yêu cầu, gom nhóm theo nguồn truy cập (```traffic_source```), sau đó lấy trung bình ```bounce_rate``` mỗi nhóm rồi sort lại để tìm ra giá trị thấp nhất là được.

In [9]:
# Đọc data web_traffic.csv
df_web_traffic = pd.read_csv('../data/raw/web_traffic.csv')
df_web_traffic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3652 entries, 0 to 3651
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      3652 non-null   object 
 1   sessions                  3652 non-null   int64  
 2   unique_visitors           3652 non-null   int64  
 3   page_views                3652 non-null   int64  
 4   bounce_rate               3652 non-null   float64
 5   avg_session_duration_sec  3652 non-null   float64
 6   traffic_source            3652 non-null   object 
dtypes: float64(2), int64(3), object(2)
memory usage: 199.8+ KB


In [28]:
# Gom nhóm theo traffic_source
traffic_analysis = df_web_traffic.groupby('traffic_source')['bounce_rate']

# Tính trung bình cộng bounce_rate
traffic_analysis = traffic_analysis.mean()

# Sort kết quả
result = traffic_analysis.sort_values(ascending=True)

print("Tỷ lệ thoát trung bình theo từng nguồn truy cập:")
print(result)
print(f"Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất: {result.index[0]},")
print(f"với tỷ lệ thoát trung bình là {result.iloc[0]:.2%}")

Tỷ lệ thoát trung bình theo từng nguồn truy cập:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64
Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất: email_campaign,
với tỷ lệ thoát trung bình là 0.45%


Dựa theo kết quả thu được, đáp án cần chọn là **C) ```email_campaign```**.

---

### Câu 5
> ***Q5**. Tỷ lệ phần trăm các dòng trong ```order_items.csv``` có áp dụng khuyến mãi (tức là ```promo_id``` **không null**) xấp xỉ là bao nhiêu?*
>
> A) 12%
>
> B) 25%
>
> C) 39%
>
> D) 54%

**Phân tích**:

Đề bảo sao làm vậy, đếm số dòng mà ```promo_id``` không null rồi chia tổng số dòng là được.

In [11]:
# Đọc data order_items.csv
df_order_items = pd.read_csv('../data/raw/order_items.csv')
df_order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  int64  
 1   product_id       714669 non-null  int64  
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         276316 non-null  object 
 6   promo_id_2       206 non-null     object 
dtypes: float64(2), int64(3), object(2)
memory usage: 38.2+ MB


In [29]:
# Lấy tổng số dòng
total_rows = len(df_order_items)

# Đếm số dòng có promo_id không null
promo_applied_rows = df_order_items['promo_id'].notnull().sum()

# Tính %
promo_percentage = (promo_applied_rows / total_rows) * 100
print(f"Tỷ lệ phần trăm các dòng có áp dụng khuyến mãi: {promo_percentage:.2f}%")

Tỷ lệ phần trăm các dòng có áp dụng khuyến mãi: 38.66%


Dựa theo kết quả thu được, đáp án xấp xỉ gần nhất là **C) 39%**.

---

### Câu 6
> ***Q6**. Trong ```customers.csv```, xét các khách hàng có ```age_group``` **khác null**, nhóm tuổi nào có **số đơn hàng trung bình trên mỗi khách hàng** cao nhất? (tổng số đơn / số khách hàng trong nhóm)*
>
> A) 55+
>
> B) 25–34
>
> C) 35–44
>
> D) 45–54

**Phân tích**:

Để làm bài này cần merge 2 bảng ```customers``` và ```orders``` theo ```customer_id```. Vì 2 bảng này đều có lượng dữ liệu lớn (vài trăm nghìn dòng), ta có thể gom các đơn hàng theo từng khách hàng trước (vì mỗi khách hàng có thể có nhiều đơn), sau đó mới merge để giảm complexity.
Sau khi merge thì ta tính tổng số đơn hàng của mỗi nhóm và từ đó tính được giá trị trung bình của mỗi nhóm để tìm được nhóm có giá trị này cao nhất.

In [13]:
# Đọc data customers.csv
df_customers = pd.read_csv('../data/raw/customers.csv')
df_customers.info()

# Check các giá trị unique của age_group
print("\nCác giá trị unique trong cột 'age_group':")
print(df_customers['age_group'].unique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   customer_id          121930 non-null  int64 
 1   zip                  121930 non-null  int64 
 2   city                 121930 non-null  object
 3   signup_date          121930 non-null  object
 4   gender               121930 non-null  object
 5   age_group            121930 non-null  object
 6   acquisition_channel  121930 non-null  object
dtypes: int64(2), object(5)
memory usage: 6.5+ MB

Các giá trị unique trong cột 'age_group':
['35-44' '45-54' '18-24' '55+' '25-34']


In [30]:
# Lọc khách hàng có age_group khác null,
# thật ra có thể không lọc vì tất cả khách hàng đều có age_group.
df_customers_filtered = df_customers[df_customers['age_group'].notnull()].copy()

# Đếm số đơn hàng của mỗi khách hàng từ bảng orders
order_counts_per_customer = df_orders.groupby('customer_id').size().reset_index(name='total_orders')

# Merge bảng khách hàng đã lọc với bảng đơn hàng
df_final = pd.merge(df_customers_filtered, order_counts_per_customer, on='customer_id', how='left')

# Điền giá trị 0 cho những khách hàng không có đơn hàng nào (loại bỏ NaN sau khi merge)
df_final['total_orders'] = df_final['total_orders'].fillna(0)

# Tính số đơn hàng trung bình cho mỗi nhóm tuổi
age_group_analysis = df_final.groupby('age_group')['total_orders'].mean()

# Sắp xếp và tìm đáp án
result = age_group_analysis.sort_values(ascending=False)

print("Số đơn hàng trung bình trên mỗi khách hàng theo nhóm tuổi:")
print(result)
print(f"Nhóm tuổi có tỷ lệ cao nhất là: {result.index[0]}")

Số đơn hàng trung bình trên mỗi khách hàng theo nhóm tuổi:
age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: total_orders, dtype: float64
Nhóm tuổi có tỷ lệ cao nhất là: 55+


Dựa theo kết quả thu được, đáp án cần tìm là **A) 55+**.

---

### Câu 7
> ***Q7**. Vùng (```region```) nào trong ```geography.csv``` tạo ra **tổng doanh thu cao nhất** trong ```sales_train.csv```?*
>
> A) West
>
> B) Central
>
> C) East
>
> D) Cả ba vùng có doanh thu xấp xỉ bằng nhau

**Phân tích**:

Bảng sales là một bảng theo kiểu snapshot, không có quan hệ trực tiếp với bảng nào khác trong tập dũ liệu, do đó không thể nối trực tiếp tới bảng geography để tính toán.

Nhận thấy để tính được sales của một ngày cần phải có thông tin về các orders và order_items trong ngày đó, 2 bảng này có quan hệ với nhau nên join được. Lại có cột zip trong bảng orders, do đó có thể join bảng orders tới bảng geography.

In [34]:
# Đọc data geography.csv và sales.csv
df_geography = pd.read_csv('../data/raw/geography.csv')
df_sales = pd.read_csv('../data/raw/sales.csv')

df_geography.info()
df_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39948 entries, 0 to 39947
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   zip       39948 non-null  int64 
 1   city      39948 non-null  object
 2   region    39948 non-null  object
 3   district  39948 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.2+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Date     3833 non-null   object 
 1   Revenue  3833 non-null   float64
 2   COGS     3833 non-null   float64
dtypes: float64(2), object(1)
memory usage: 90.0+ KB


In [35]:
# Tính doanh thu cho từng dòng sản phẩm trong order_items
# Doanh thu = (Số lượng * Đơn giá) - Giảm giá
df_order_items['line_revenue'] = (df_order_items['quantity'] * df_order_items['unit_price']) - df_order_items['discount_amount']

# 3. Kết hợp order_items với orders để lấy mã zip
df_order_with_zip = pd.merge(df_order_items[['order_id', 'line_revenue']], 
                    df_orders[['order_id', 'zip', 'order_status']], 
                    on='order_id', how='inner')

# 4. Lọc bỏ các đơn hàng bị hủy (cancelled) để số liệu chính xác với "doanh thu thuần"
df_order_with_zip = df_order_with_zip[df_order_with_zip['order_status'] != 'cancelled']

# 5. Kết hợp với geography để lấy vùng (region)
df_order_with_geo = pd.merge(df_order_with_zip, df_geography[['zip', 'region']], on='zip', how='inner')

# 6. Tổng hợp doanh thu theo vùng
region_revenue = df_order_with_geo.groupby('region')['line_revenue'].sum().sort_values(ascending=False)

# 7. Tính tỷ lệ phần trăm để kiểm tra đáp án D (xấp xỉ bằng nhau)
total_revenue = region_revenue.sum()
region_percentage = (region_revenue / total_revenue) * 100

print("Tổng doanh thu theo từng vùng:")
print(region_revenue)
print("\nTỷ lệ phần trăm doanh thu giữa các vùng:")
print(region_percentage)

Tổng doanh thu theo từng vùng:
region
East       6.617596e+09
Central    4.278239e+09
West       3.338021e+09
Name: line_revenue, dtype: float64

Tỷ lệ phần trăm doanh thu giữa các vùng:
region
East       46.491941
Central    30.056781
West       23.451277
Name: line_revenue, dtype: float64


Dựa theo kết quả thu được, ta thấy vùng East chiếm gần một nửa, hơn đáng kể 2 vùng còn lại.

Đáp án cần tìm là **C) East**.

---

### Câu 8
> ***Q8**. Trong các đơn hàng có ```order_status = ’cancelled’``` trong ```orders.csv```, **phương thức thanh toán** nào được sử dụng nhiều nhất?*
>
> A) ```credit_card```
>
> B) ```cod```
>
> C) ```paypal```
>
> D) ```bank_transfer```

**Phân tích**:

Đề bảo sao làm nấy, lọc các đơn có ```order_status = ’cancelled’``` rồi đếm số lần xuất hiện của mỗi loại thanh toán để tìm ra lựa chọn được sử dụng nhiều nhất là được.

In [17]:
# check các giá trị unique của cột order_status trong df_orders
print("Các giá trị unique trong cột 'order_status':")
print(df_orders['order_status'].unique())

Các giá trị unique trong cột 'order_status':
['delivered' 'returned' 'paid' 'cancelled' 'created' 'shipped']


In [ ]:
# Tìm các đơn hàng 'cancelled'
cancelled_orders = df_orders[df_orders['order_status'] == 'cancelled']

# Đếm tần suất các payment_method
payment_counts = cancelled_orders['payment_method'].value_counts()

print("Thống kê phương thức thanh toán của các đơn hàng bị hủy:")
print(payment_counts)

Thống kê phương thức thanh toán của các đơn hàng bị hủy:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64


Dựa theo kết quả thu được, đáp án cần tìm là **A) ```credit_card```**.

---

### Câu 9
> ***Q9**. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có **tỷ lệ trả hàng cao nhất**, được định nghĩa là số bản ghi trong ```returns``` chia cho số dòng trong ```order_items``` (**join** với ```products``` theo ```product_id```)?*
>
> A) S
>
> B) M
>
> C) L
>
> D) XL

**Phân tích**:

Ta cần tính số dòng trong returns của mỗi loại kích thước sản phẩm chia cho số dòng tổng của loại kích thước đó trong order_items. 
Để tính số dòng tổng của mỗi loại, ta join bảng order_items với products và lọc lấy 4 size như đề bài.
Để tính số dòng trong returns của mỗi loại, ta join bảng returns với products và cũng lọc tương tự.
Sau khi có 2 tập giá trị này, ta có thể tính tỉ lệ trả hàng của từng loại.

In [32]:
target_sizes = ['S', 'M', 'L', 'XL']

# Tính tổng số dòng trong order_items theo từng size
# join order_items với products
df_orders_prod = pd.merge(df_order_items, df_products[['product_id', 'size']], on='product_id', how='left')
# Lọc 4 size
df_orders_prod = df_orders_prod[df_orders_prod['size'].isin(target_sizes)]
# Đếm số dòng cho mỗi size
order_counts = df_orders_prod.groupby('size').size()

# Tính số dòng trong returns theo từng size
# join returns với products
df_returns_prod = pd.merge(df_returns, df_products[['product_id', 'size']], on='product_id', how='left')
# Lọc 4 size
df_returns_prod = df_returns_prod[df_returns_prod['size'].isin(target_sizes)]
# Đếm số dòng cho mỗi size
return_counts = df_returns_prod.groupby('size').size()

# Tính tỷ lệ trả hàng
return_rate = (return_counts / order_counts).sort_values(ascending=False)

print("Tỷ lệ trả hàng theo từng kích thước (Return Rate):")
print(return_rate)
print(f"Kích thước có tỷ lệ trả hàng cao nhất: {return_rate.index[0]},")
print(f"với tỷ lệ trả hàng là {return_rate.iloc[0]:.2%}")

Tỷ lệ trả hàng theo từng kích thước (Return Rate):
size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
dtype: float64
Kích thước có tỷ lệ trả hàng cao nhất: S,
với tỷ lệ trả hàng là 5.65%


Dựa theo kết quả thu được, đáp án cần chọn là **A) S**.

---

### Câu 10
> ***Q10**. Trong ```payments.csv```, **kế hoạch trả góp** nào có **giá trị thanh toán trung bình trên mỗi đơn hàng** cao nhất?*
>
> A) 1 kỳ (trả một lần)
>
> B) 3 kỳ
> 
> C) 6 kỳ
>
> D) 12 kỳ

**Phân tích**:

Theo bảng 2 trong đề bài, ràng buộc giữa orders $\leftrightarrow$ payments là 1:1, nghĩa là mỗi đơn hàng chỉ có 1 bản ghi thanh toán tương ứng duy nhất. Do đó, không cần merge với bảng orders ta vẫn có thể tính được giá trị thanh toán trung bình của mỗi đơn hàng, ở đây chính là tính trên cột ```payment_value```, sau đó lọc và gom theo các nhóm với mỗi nhóm là số kỳ trả góp là có thể tìm được đáp án.

In [21]:
# Đọc data payments.csv
df_payments = pd.read_csv('../data/raw/payments.csv')
df_payments.info()

# Check các giá trị unique của cột installments
print("\nCác giá trị unique trong cột 'installments':")
print(df_payments['installments'].unique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   order_id        646945 non-null  int64  
 1   payment_method  646945 non-null  object 
 2   payment_value   646945 non-null  float64
 3   installments    646945 non-null  int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 19.7+ MB

Các giá trị unique trong cột 'installments':
[ 3  1 12  6  2]


In [31]:
# Gom nhóm theo installments sau đó tính trung bình cộng của giá trị thanh toán (payment_value)
installment_analysis = df_payments.groupby('installments')['payment_value'].mean()

# Lọc giữ lại các mốc trong đề (1, 3, 6, 12)
target_installments = [1, 3, 6, 12]
# reindex để chỉ lấy những giá trị cần thiết
result = installment_analysis.reindex(target_installments).sort_values(ascending=False)

print("Giá trị thanh toán trung bình theo kế hoạch trả góp:")
print(result)
print(f"Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất: {result.index[0]} kỳ,")
print(f"với giá trị thanh toán trung bình là {result.iloc[0]:.2f}")

Giá trị thanh toán trung bình theo kế hoạch trả góp:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
Name: payment_value, dtype: float64
Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất: 6 kỳ,
với giá trị thanh toán trung bình là 24446.65


Dựa theo kết quả thu được, đáp án thu được là **C) 6 kỳ**.

---